# Exploratory Data Analysis — AI Logistics EDD Control Tower

This notebook is the exploratory counterpart to the production pipeline in `src/`. It re-derives the headline numbers from `data/processed/shipments_features.csv` (run `python run_pipeline.py` from the repo root first) so you can sanity-check every claim in `docs/business_impact.md` interactively.

Nothing here is fabricated: every cell either reads an ACTUAL field from the source workbook or a DERIVED/SYNTHETIC/AI_PREDICTED field documented in `docs/data_dictionary.md`.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from config.config import FEATURED_SHIPMENTS_PATH

df = pd.read_csv(FEATURED_SHIPMENTS_PATH, parse_dates=['order_date','pickup_date','delivery_date','edd'])
print(f'{len(df)} shipments, {df.shape[1]} columns')
df.head()

## 1. Baseline EDD adherence (ACTUAL)

Denominator is delivered shipments only, matching the source workbook's own Validation sheet (1546/1819 = 84.99%).

In [ ]:
delivered = df[df['is_delivered']]
baseline = delivered['edd_met'].sum() / len(delivered)
print(f"Delivered shipments: {len(delivered)}")
print(f"EDD met: {delivered['edd_met'].sum()}")
print(f"Baseline EDD adherence: {baseline:.2%}")
print(f"Gap to 94% target: {(0.94 - baseline)*100:.1f} pp")

## 2. Status mix and failure modes

In [ ]:
status_counts = df['current_status'].value_counts()
print(status_counts)
status_counts.plot(kind='bar', title='Shipments by Current Status', figsize=(7,4))
plt.ylabel('Shipments')
plt.tight_layout()
plt.show()

## 3. COD vs Prepaid RTO behaviour

In [ ]:
rto_by_payment = df.groupby('payment_mode')['is_rto'].mean() * 100
print(rto_by_payment.round(1))
rto_by_payment.plot(kind='bar', color=['#2a78d6','#eb6834'], title='RTO % by Payment Mode', figsize=(5,4))
plt.ylabel('RTO %')
plt.tight_layout()
plt.show()

## 4. NDR Pareto (ACTUAL reasons)

In [ ]:
ndr_counts = df.loc[df['has_ndr'], 'ndr_reason'].value_counts()
cum_pct = ndr_counts.cumsum() / ndr_counts.sum() * 100
pd.DataFrame({'count': ndr_counts, 'cumulative_pct': cum_pct.round(1)})

## 5. Lane class performance (DERIVED geography)

In [ ]:
lane_perf = delivered.groupby('lane_class')['edd_met'].agg(['mean','count'])
lane_perf['edd_adherence_pct'] = (lane_perf['mean']*100).round(1)
lane_perf[['edd_adherence_pct','count']].sort_values('edd_adherence_pct')

## 6. Weekly EDD adherence trend

In [ ]:
weekly = delivered.copy()
weekly['order_week'] = weekly['order_date'].dt.to_period('W').apply(lambda p: p.start_time)
trend = weekly.groupby('order_week')['edd_met'].mean() * 100
trend.plot(marker='o', figsize=(8,4), title='EDD Adherence Trend (weekly, ACTUAL)')
plt.axhline(94, color='#eda100', linestyle='--', label='Target 94%')
plt.ylabel('EDD Adherence %')
plt.legend()
plt.tight_layout()
plt.show()

## Next steps

For the full agent outputs (lane scorecards, carrier recommendations, NDR queue, COD remittance queue, ML risk predictions, and the 85%→94% intervention simulation), see the `outputs/` directory generated by `python run_pipeline.py`, or open `dashboard/index.html` for the executive view.